# Compare CIFAKE model accuracy (Google Colab)

This notebook evaluates the existing **CIFAKE Custom CNN** checkpoint and the app's pinned **Hugging Face ViT** on the exact same held-out CIFAKE `test/REAL` and `test/FAKE` images. It does **not** train or fine-tune either model. Run cells from top to bottom.

## Before you start

1. In Colab, choose **Runtime → Change runtime type → T4 GPU** (optional, but faster).
2. Upload the CIFAKE ZIP and your trained CNN `.keras` checkpoint to `/content`, or edit the paths in the setup cell to locations in mounted Google Drive.
   - Dataset ZIP must contain a `test/REAL/` and `test/FAKE/` folder. An optional parent directory inside the ZIP is okay.
   - Use the same `cifake_custom_cnn.keras` checkpoint that the app uses (or the matching best checkpoint from the same training run).
3. Run all cells. The ViT checkpoint is downloaded from Hugging Face at its pinned revision.

Both models see the full image, with their own model-specific input preprocessing: the CNN resizes RGB to 32×32 with bilinear interpolation and scales pixels to [0, 1]; the ViT uses its Hugging Face image processor (224×224). `REAL` is label 0, `FAKE` is label 1, and both use a 0.5 AI-score threshold. Results only describe this CIFAKE test set; they do not establish accuracy on arbitrary real photos or images from newer generators.

## 1. Install and import evaluation dependencies

In [ ]:
%pip -q install "transformers>=4.39.3,<5" scikit-learn pandas matplotlib safetensors

import json
import os
import zipfile
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import torch
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from transformers import AutoImageProcessor, AutoModelForImageClassification

for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

MODEL_ID = "dima806/ai_vs_real_image_detection"
MODEL_REVISION = "5d647683672ffd0080441f4e8b4345934c44cc61"
THRESHOLD = 0.5
CNN_BATCH_SIZE = 128
VIT_BATCH_SIZE = 16
print(f"TensorFlow: {tf.__version__}; PyTorch: {torch.__version__}")
print(f"TensorFlow GPU(s): {tf.config.list_physical_devices('GPU')}")
print(f"PyTorch device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Locate the dataset ZIP and CNN checkpoint

If the files are in Google Drive, mount Drive first in a separate cell with `from google.colab import drive; drive.mount('/content/drive')`, then set the two paths below. Leave them at the defaults to select the files in an upload dialog.

In [ ]:
from google.colab import files

# Edit these paths if your files are in Google Drive or another folder.
DATASET_ZIP = Path("/content/archive (2).zip")
CNN_MODEL_PATH = Path("/content/cifake_custom_cnn.keras")

missing_paths = [path for path in (DATASET_ZIP, CNN_MODEL_PATH) if not path.is_file()]
if missing_paths:
    print("Upload the CIFAKE dataset ZIP and the trained CNN .keras checkpoint.")
    uploaded = files.upload()
    for filename in uploaded:
        candidate = Path("/content") / filename
        if candidate.suffix.lower() == ".zip" and not DATASET_ZIP.is_file():
            DATASET_ZIP = candidate
        elif candidate.suffix.lower() == ".keras" and not CNN_MODEL_PATH.is_file():
            CNN_MODEL_PATH = candidate

if not DATASET_ZIP.is_file():
    raise FileNotFoundError(f"CIFAKE dataset ZIP not found: {DATASET_ZIP}")
if not CNN_MODEL_PATH.is_file():
    raise FileNotFoundError(f"CIFAKE CNN checkpoint not found: {CNN_MODEL_PATH}")
print(f"Dataset: {DATASET_ZIP} ({DATASET_ZIP.stat().st_size / (1024**2):.1f} MiB)")
print(f"CNN checkpoint: {CNN_MODEL_PATH} ({CNN_MODEL_PATH.stat().st_size / (1024**2):.1f} MiB)")

## 3. Validate the held-out test split

The notebook reads image bytes directly from the ZIP and only evaluates entries under `test/REAL/` or `test/FAKE/`. It does not extract or modify the archive.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}
CLASS_LABELS = {"REAL": 0, "FAKE": 1}

with zipfile.ZipFile(DATASET_ZIP) as archive:
    test_members = []
    for member in archive.infolist():
        if member.is_dir():
            continue
        parts = Path(member.filename).parts
        if Path(member.filename).suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        for index, part in enumerate(parts[:-2]):
            if part.lower() == "test" and parts[index + 1].upper() in CLASS_LABELS:
                class_name = parts[index + 1].upper()
                test_members.append((member, CLASS_LABELS[class_name], class_name))
                break

if not test_members:
    raise ValueError(
        "No test images found. Expected test/REAL and test/FAKE folders in the ZIP."
    )
class_counts = pd.Series([item[2] for item in test_members]).value_counts().to_dict()
if set(class_counts) != set(CLASS_LABELS) or min(class_counts.values()) == 0:
    raise ValueError(f"Both REAL and FAKE test classes are required; found {class_counts}.")
if len({item[0].filename for item in test_members}) != len(test_members):
    raise ValueError("Duplicate test image paths were found in the ZIP.")
print(f"Held-out images: {len(test_members):,}")
print(f"Class counts: {class_counts}")

## 4. Evaluate the CIFAKE Custom CNN

This applies the same RGB conversion, 32×32 bilinear resize, and [0, 1] scaling as the app.

In [ ]:
def read_rgb_batch(archive, batch_members):
    """Decode one batch and fail explicitly if any test image is unreadable."""
    images = []
    labels = []
    for member, label, _ in batch_members:
        try:
            with archive.open(member) as image_file:
                with Image.open(image_file) as image:
                    images.append(image.convert("RGB"))
        except (OSError, UnidentifiedImageError, zipfile.BadZipFile) as error:
            raise RuntimeError(f"Could not decode {member.filename}: {error}") from error
        labels.append(label)
    return images, np.asarray(labels, dtype=np.int64)


def calculate_metrics(y_true, ai_scores):
    """Calculate binary metrics using AI-generated (FAKE) as positive class."""
    y_true = np.asarray(y_true, dtype=np.int64)
    ai_scores = np.asarray(ai_scores, dtype=np.float64)
    if len(y_true) != len(ai_scores) or not np.isfinite(ai_scores).all():
        raise ValueError("Labels and finite model scores must have matching lengths.")
    if (ai_scores < 0).any() or (ai_scores > 1).any():
        raise ValueError("Model AI scores must be probabilities in [0, 1].")
    predictions = (ai_scores >= THRESHOLD).astype(np.int64)
    matrix = confusion_matrix(y_true, predictions, labels=[0, 1])
    return {
        "n": int(len(y_true)),
        "real_count": int(np.sum(y_true == 0)),
        "fake_count": int(np.sum(y_true == 1)),
        "threshold": THRESHOLD,
        "accuracy": float(accuracy_score(y_true, predictions)),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "f1_score": float(f1_score(y_true, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, ai_scores)),
        "confusion_matrix_labels_0_1": matrix.tolist(),
    }


cnn_model = tf.keras.models.load_model(CNN_MODEL_PATH, compile=False)
cnn_labels = []
cnn_scores = []
with zipfile.ZipFile(DATASET_ZIP) as archive:
    for start in range(0, len(test_members), CNN_BATCH_SIZE):
        batch = test_members[start : start + CNN_BATCH_SIZE]
        images, labels = read_rgb_batch(archive, batch)
        resized = [
            np.asarray(
                image.resize((32, 32), resample=Image.Resampling.BILINEAR),
                dtype=np.float32,
            ) / 255.0
            for image in images
        ]
        output = np.asarray(cnn_model(np.stack(resized), training=False))
        if output.ndim == 2 and output.shape[1] == 1:
            ai_probabilities = output[:, 0]
        elif output.ndim == 2 and output.shape[1] == 2:
            ai_probabilities = output[:, 1]
        else:
            raise ValueError(f"Unexpected CNN output shape: {output.shape}")
        cnn_labels.extend(labels.tolist())
        cnn_scores.extend(np.clip(ai_probabilities, 0.0, 1.0).tolist())
        completed = min(start + len(batch), len(test_members))
        if completed % 2000 < len(batch) or completed == len(test_members):
            print(f"CNN evaluated {completed:,}/{len(test_members):,}")

cnn_results = calculate_metrics(cnn_labels, cnn_scores)
print(json.dumps(cnn_results, indent=2))
del cnn_model
tf.keras.backend.clear_session()

## 5. Evaluate the pinned Hugging Face ViT

The model processor gets each complete RGB image and performs its checkpoint-specific preprocessing. Inference is batched and uses the GPU when Colab provides one, otherwise CPU.

In [ ]:
processor = AutoImageProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
vit_model = AutoModelForImageClassification.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    use_safetensors=True,
)
vit_labels = {
    int(index): str(label).strip().upper()
    for index, label in vit_model.config.id2label.items()
}
fake_ids = [index for index, label in vit_labels.items() if label == "FAKE"]
if len(fake_ids) != 1:
    raise ValueError(f"Expected exactly one FAKE output class; model labels: {vit_labels}")
fake_class_id = fake_ids[0]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vit_model.to(device).eval()
print(f"Model: {MODEL_ID}@{MODEL_REVISION}")
print(f"Output labels: {vit_labels}; device: {device}")

vit_y_true = []
vit_scores = []
with zipfile.ZipFile(DATASET_ZIP) as archive:
    for start in range(0, len(test_members), VIT_BATCH_SIZE):
        batch = test_members[start : start + VIT_BATCH_SIZE]
        images, labels = read_rgb_batch(archive, batch)
        inputs = processor(images=images, return_tensors="pt")
        inputs = {name: tensor.to(device) for name, tensor in inputs.items()}
        with torch.inference_mode():
            logits = vit_model(**inputs).logits
            probabilities = torch.softmax(logits, dim=-1)[:, fake_class_id]
        vit_y_true.extend(labels.tolist())
        vit_scores.extend(probabilities.detach().cpu().tolist())
        completed = min(start + len(batch), len(test_members))
        if completed % 1000 < len(batch) or completed == len(test_members):
            print(f"ViT evaluated {completed:,}/{len(test_members):,}")

vit_results = calculate_metrics(vit_y_true, vit_scores)
print(json.dumps(vit_results, indent=2))

## 6. Compare, visualize, and save results

Accuracy is directly comparable here because both models were run on the same CIFAKE test entries and labels. It remains a CIFAKE-only comparison, not a general real-world ranking.

In [ ]:
if cnn_results["n"] != vit_results["n"] or cnn_results["n"] != len(test_members):
    raise RuntimeError("The models were not evaluated on the same complete test set.")

comparison = pd.DataFrame(
    [
        {"model": "CIFAKE Custom CNN", **cnn_results},
        {"model": "Hugging Face ViT", **vit_results},
    ]
)[["model", "n", "real_count", "fake_count", "accuracy", "precision", "recall", "f1_score", "roc_auc", "threshold"]]
display(comparison.style.format({
    "accuracy": "{:.2%}", "precision": "{:.2%}", "recall": "{:.2%}",
    "f1_score": "{:.2%}", "roc_auc": "{:.4f}", "threshold": "{:.2f}"
}))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for axis, name, result in zip(
    axes,
    ("CIFAKE Custom CNN", "Hugging Face ViT"),
    (cnn_results, vit_results),
):
    matrix = np.asarray(result["confusion_matrix_labels_0_1"])
    axis.imshow(matrix, cmap="Blues")
    axis.set_title(f"{name}\nAccuracy: {result['accuracy']:.2%}")
    axis.set_xlabel("Predicted label")
    axis.set_ylabel("True label")
    axis.set_xticks([0, 1], labels=["REAL", "FAKE"])
    axis.set_yticks([0, 1], labels=["REAL", "FAKE"])
    for (row, column), value in np.ndenumerate(matrix):
        axis.text(column, row, str(value), ha="center", va="center")
fig.tight_layout()
plt.show()

RESULTS_DIR = Path("/content/cifake_model_comparison_results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
metadata = {
    "dataset": "CIFAKE held-out test split",
    "dataset_zip": DATASET_ZIP.name,
    "sample_count": len(test_members),
    "label_mapping": CLASS_LABELS,
    "threshold": THRESHOLD,
    "huggingface_model": MODEL_ID,
    "huggingface_revision": MODEL_REVISION,
    "cnn_checkpoint": CNN_MODEL_PATH.name,
    "preprocessing": {
        "cnn": "RGB bilinear resize to 32x32; float32 values scaled to [0,1]",
        "huggingface_vit": "Pinned Hugging Face image processor; full RGB image",
    },
    "results": {
        "cifake_custom_cnn": cnn_results,
        "huggingface_vit": vit_results,
    },
    "limitation": (
        "Metrics apply only to this CIFAKE test split and do not establish accuracy "
        "on arbitrary real-world images or current image generators."
    ),
}
(RESULTS_DIR / "comparison_results.json").write_text(
    json.dumps(metadata, indent=2) + "\n", encoding="utf-8"
)
comparison.to_csv(RESULTS_DIR / "comparison_summary.csv", index=False)
print(f"Saved results to {RESULTS_DIR}")

## 7. Download the comparison results

In [ ]:
import shutil
from google.colab import files

RESULTS_ZIP = Path("/content/cifake_model_comparison_results.zip")
shutil.make_archive(str(RESULTS_ZIP.with_suffix("")), "zip", RESULTS_DIR)
print(f"Created {RESULTS_ZIP}")
files.download(str(RESULTS_ZIP))